ตั้งค่าระบบ การเข้า GEE & Python

In [1]:
!pip install earthengine-api --upgrade
!pip install folium
!pip install pyproj

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.5/480.5 kB 4.9 MB/s eta 0:00:00
  Attempting uninstall: earthengine-api
    Found existing installation: earthengine-api 1.7.32
    Uninstalling earthengine-api-1.7.32:
      Successfully uninstalled earthengine-api-1.7.32


In [2]:
import ee
import folium
ee.Authenticate()
ee.Initialize(project='ee-kulchiranut')

ทำการโค้ดวิเคราะห์

In [3]:
import ee
import folium

# ==============================================================================
# STEP 1: นำเข้าพื้นที่ศึกษา (ROI) และตั้งค่าแผนที่เบื้องต้น
# ==============================================================================
lat_min, lon_min = 17.880568, 102.722825
lat_max, lon_max = 17.875089, 102.722825

roi = ee.Geometry.Polygon([
    [[102.49856452615988, 17.725056403180492],
     [102.91329841287863, 17.725056403180492],
     [102.91329841287863, 18.047864078632685],
     [102.49856452615988, 18.047864078632685],
     [102.49856452615988, 17.725056403180492]]
])

Map = folium.Map(location=[(lat_min + lat_max) / 2, (lon_min + lon_max) / 2], zoom_start=9)

# แผนที่ฐาน (Base Map)
folium.TileLayer(
    tiles='https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}',
    attr='Google Satellite',
    name='Google Satellite',
    overlay=False,
    control=True
).add_to(Map)

# ==============================================================================
# STEP 2: การกำหนดรูปแบบการแสดงสี (Visualization Parameters)
# ==============================================================================
vis_roi = {'color': 'FFFF00', 'fillColor': '00000000', 'width': 2}
ndwi_palette = ['#a52a2a', '#ffffff', '#0000ff']
erosion_palette = ['#3333FF', '#0070FF', '#00BFFF', '#00FF00', '#FFFF00', '#FFA500', '#FF0000']

vis_params_l5 = {'bands': ['SR_B3', 'SR_B2', 'SR_B1'], 'min': 0.0, 'max': 0.3}
vis_params_l9 = {'bands': ['SR_B4', 'SR_B3', 'SR_B2'], 'min': 0.0, 'max': 0.3}
vis_ndwi = {'min': -1, 'max': 1, 'palette': ndwi_palette}
vis_change = {'min': -0.4, 'max': 0.4, 'palette': erosion_palette}

# ==============================================================================
# STEP 3: ฟังก์ชันสำหรับการเตรียมข้อมูลดาวเทียม (ดึงภาพ + Scale + คำนวณ NDWI)
# ==============================================================================
def process_landsat5(date_start, date_end):
    collection = ee.ImageCollection("LANDSAT/LT05/C02/T1_L2") \
        .filterBounds(roi) \
        .filterDate(date_start, date_end) \
        .sort('CLOUD_COVER')
    image = collection.first()

    def apply_scale_factors(img):
        optical_bands = img.select('SR_B.').multiply(0.0000275).add(-0.2)
        thermal_bands = img.select('ST_B6').multiply(0.00341802).add(149.0)
        return img.addBands(optical_bands, None, True).addBands(thermal_bands, None, True)

    image = apply_scale_factors(image)
    ndwi = image.normalizedDifference(['SR_B2', 'SR_B4']).rename('NDWI')
    return image, ndwi

def process_landsat9(date_start, date_end):
    collection = ee.ImageCollection("LANDSAT/LC09/C02/T1_L2") \
        .filterBounds(roi) \
        .filterDate(date_start, date_end) \
        .sort('CLOUD_COVER')
    image = collection.first()

    def apply_scale_factors(img):
        optical_bands = img.select('SR_B.').multiply(0.0000275).add(-0.2)
        thermal_bands = img.select('ST_B10').multiply(0.00341802).add(149.0)
        return img.addBands(optical_bands, None, True).addBands(thermal_bands, None, True)

    image = apply_scale_factors(image)
    ndwi = image.normalizedDifference(['SR_B3', 'SR_B5']).rename('NDWI')
    return image, ndwi

# ==============================================================================
# STEP 4: ประมวลผลและเตรียมข้อมูล (ยังไม่ใส่ลงแผนที่)
# ==============================================================================
img_1993, ndwi_1993 = process_landsat5('1993-01-01', '1993-12-31')
img_2003, ndwi_2003 = process_landsat5('2003-01-01', '2003-12-31')
img_2026, ndwi_2026 = process_landsat9('2026-01-01', '2026-07-10')

# คำนวณการเปลี่ยนแปลงการกัดเซาะ/ทับถม
change_93_03 = ndwi_2003.subtract(ndwi_1993)
change_03_26 = ndwi_2026.subtract(ndwi_2003)
change_93_26 = ndwi_2026.subtract(ndwi_1993)

# ==============================================================================
# STEP 5: นำเข้าเลเยอร์สู่แผนที่ตามลำดับที่คุณต้องการ (1 - 9)
# ==============================================================================

# --- กลุ่มภาพสีจริง (True Color) ---
# 1. สีจริงปี 1993
folium.TileLayer(
    tiles=img_1993.getMapId(vis_params_l5)['tile_fetcher'].url_format,
    attr='GEE - Landsat 5', name='1. Landsat 5 ปี 1993 (True Color)', overlay=True, control=True, show=False
).add_to(Map)

# 2. สีจริงปี 2003
folium.TileLayer(
    tiles=img_2003.getMapId(vis_params_l5)['tile_fetcher'].url_format,
    attr='GEE - Landsat 5', name='2. Landsat 5 ปี 2003 (True Color)', overlay=True, control=True, show=False
).add_to(Map)

# 3. สีจริงปี 2026
folium.TileLayer(
    tiles=img_2026.getMapId(vis_params_l9)['tile_fetcher'].url_format,
    attr='GEE - Landsat 9', name='3. Landsat 9 ปี 2026 (True Color)', overlay=True, control=True, show=False
).add_to(Map)


# --- กลุ่มการวิเคราะห์ NDWI ---
# 4. การวิเคราะห์ NDWI ปี 1993
folium.TileLayer(
    tiles=ndwi_1993.getMapId(vis_ndwi)['tile_fetcher'].url_format,
    attr='GEE - NDWI', name='4. Landsat 5 ปี 1993 (NDWI)', overlay=True, control=True, show=False
).add_to(Map)

# 5. การวิเคราะห์ NDWI ปี 2003
folium.TileLayer(
    tiles=ndwi_2003.getMapId(vis_ndwi)['tile_fetcher'].url_format,
    attr='GEE - NDWI', name='5. Landsat 5 ปี 2003 (NDWI)', overlay=True, control=True, show=False
).add_to(Map)

# 6. การวิเคราะห์ NDWI ปี 2026
folium.TileLayer(
    tiles=ndwi_2026.getMapId(vis_ndwi)['tile_fetcher'].url_format,
    attr='GEE - NDWI', name='6. Landsat 9 ปี 2026 (NDWI)', overlay=True, control=True, show=False
).add_to(Map)


# --- กลุ่มการวิเคราะห์การเปลี่ยนแปลง (Change Detection) ---
# 7. การวิเคราะห์การเปลี่ยนแปลง 1993 - 2003
folium.TileLayer(
    tiles=change_93_03.getMapId(vis_change)['tile_fetcher'].url_format,
    attr='GEE - Change 1993-2003', name='7. วิเคราะห์การกัดเซาะ/ทับถม (1993-2003)', overlay=True, control=True, show=False
).add_to(Map)

# 8. การวิเคราะห์การเปลี่ยนแปลง 2003 - 2026
folium.TileLayer(
    tiles=change_03_26.getMapId(vis_change)['tile_fetcher'].url_format,
    attr='GEE - Change 2003-2026', name='8. วิเคราะห์การกัดเซาะ/ทับถม (2003-2026)', overlay=True, control=True, show=False
).add_to(Map)

# 9. การวิเคราะห์การเปลี่ยนแปลงภาพรวม 1993 - 2026
folium.TileLayer(
    tiles=change_93_26.getMapId(vis_change)['tile_fetcher'].url_format,
    attr='GEE - Change 1993-2026', name='9. วิเคราะห์การกัดเซาะ/ทับถมรวม (1993-2026)', overlay=True, control=True, show=True
).add_to(Map)


# --- ขอบเขตพื้นที่ศึกษา (ROI) แดง/เหลือง ไว้บนสุด ---
Map.add_child(folium.raster_layers.TileLayer(
    tiles=ee.Image().paint(roi, 0, 3).getMapId(vis_roi)['tile_fetcher'].url_format,
    attr='Google Earth Engine', name='ขอบเขตพื้นที่ศึกษา', overlay=True, control=True
))

# แสดงกล่องควบคุมเลเยอร์
folium.LayerControl(position='topright', collapsed=False).add_to(Map)

Map

ทำการกำหนดเฉพาะพื้นที่ศึกษาเนื่องจากเตรียมการส่งออกที่ไม่นาน

In [4]:
#ทำการ Clip เลเยอร์การเปลี่ยนแปลง (Change Detection)
change_93_03_clipped = change_93_03.clip(roi)
change_03_26_clipped = change_03_26.clip(roi)
change_93_26_clipped = change_93_26.clip(roi)

#สร้างแผนที่ใหม่เพื่อแสดงผลเฉพาะพื้นที่ที่ Clip แล้ว
Map = folium.Map(location=[17.880568, 102.722825], zoom_start=11)

# เพิ่ม Satellite Base
folium.TileLayer(
    tiles='https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}',
    attr='Google Satellite',
    name='Google Satellite',
    overlay=False
).add_to(Map)

# ฟังก์ชันช่วยเพิ่ม Layer ที่ Clip แล้ว
def add_clipped_layer(ee_image, vis, name, show=False):
    map_id = ee_image.getMapId(vis)
    folium.TileLayer(
        tiles=map_id['tile_fetcher'].url_format,
        attr='GEE',
        name=name,
        overlay=True,
        show=show
    ).add_to(Map)

# แสดงการกัดเซาะ/ทับถม (Clipped)
add_clipped_layer(change_93_03_clipped, vis_change, 'การกัดเซาะ/ทับถม 1993-2003 (Clipped)', show=True)
add_clipped_layer(change_03_26_clipped, vis_change, 'การกัดเซาะ/ทับถม 2003-2026 (Clipped)', show=True)
add_clipped_layer(change_93_26_clipped, vis_change, 'การกัดเซาะ/ทับถม 1993-2026 (Clipped)', show=True)

# แสดงขอบเขต ROI
folium.GeoJson(roi.getInfo(), name='ROI Polygon', style_function=lambda x: {'color': 'yellow', 'fillColor': '#00000000'}).add_to(Map)

folium.LayerControl().add_to(Map)
Map

แปลง Raster เป็น Vector

In [6]:
# ==============================================================================
# STEP 6: [ปรับปรุง] จัดกลุ่มข้อมูล, แอดลงแผนที่ และสั่งดาวน์โหลดลงคอมพิวเตอร์ทันที
# ==============================================================================
import requests

# ตรวจสอบระบบว่ารันบน Google Colab หรือไม่ เพื่อใช้คำสั่งดาวน์โหลดอัตโนมัติ
try:
    from google.colab import files
    USING_COLAB = True
except ImportError:
    USING_COLAB = False

def export_display_and_download_direct(change_image, filename_prefix, display_name):
    """
    ฟังก์ชันจำแนกกลุ่มข้อมูล -> แอดลงแผนที่ -> โหลดไฟล์เบื้องหลัง -> เด้งดาวน์โหลดลงคอมพิวเตอร์ (.zip)
    """
    # 1. จัดกลุ่มค่าทศนิยมให้เป็นเลขกลุ่ม (Integer 1-7)
    classified_image = ee.Image(0) \
        .where(change_image.lte(-0.28), 1) \
        .where(change_image.gt(-0.28).And(change_image.lte(-0.16)), 2) \
        .where(change_image.gt(-0.16).And(change_image.lte(-0.04)), 3) \
        .where(change_image.gt(-0.04).And(change_image.lte(0.04)), 4) \
        .where(change_image.gt(0.04).And(change_image.lte(0.16)), 5) \
        .where(change_image.gt(0.16).And(change_image.lte(0.28)), 6) \
        .where(change_image.gt(0.28), 7) \
        .clip(roi)

    # 2. แปลงจาก Raster ให้กลายเป็น Vector Polygons
    vector_polygons = classified_image.reduceToVectors(
        geometry=roi,
        scale=30,
        geometryType='polygon',
        eightConnected=True,
        labelProperty='zone_class',
        maxPixels=1e8
    )

    # 3. เพิ่มเลเยอร์พื้นที่กลุ่มสีลงบนแผนที่
    folium.TileLayer(
        tiles=classified_image.getMapId({'min': 1, 'max': 7, 'palette': erosion_palette})['tile_fetcher'].url_format,
        attr='GEE Classified',
        name=f'🎨 พื้นที่จำแนกกลุ่ม {display_name}',
        overlay=True,
        show=False
    ).add_to(Map)

    # 4. เพิ่มเลเยอร์เส้นขอบ Vector ลงบนแผนที่
    vector_styled = vector_polygons.style(color='000000', width=1, fillColor='00000000')
    folium.TileLayer(
        tiles=vector_styled.getMapId()['tile_fetcher'].url_format,
        attr='GEE Vector Outline',
        name=f'🕸️ เส้นขอบ Vector {display_name}',
        overlay=True,
        show=False
    ).add_to(Map)

    # 5. สั่งระบบสร้างลิงก์ดาวน์โหลดภายใน (ซ่อนไว้ ไม่แสดงผลออกมาให้เห็น)
    download_url = vector_polygons.getDownloadURL(
        filetype='SHP',
        selectors=['zone_class'],
        filename=filename_prefix
    )

    # 6. ดาวน์โหลดไฟล์จาก Server GEE มาเขียนเป็นไฟล์ .zip ชั่วคราวในโปรแกรม
    zip_filename = f"{filename_prefix}.zip"
    response = requests.get(download_url)
    with open(zip_filename, 'wb') as f:
        f.write(response.content)

    # 7. สั่งให้ไฟล์เด้งดาวน์โหลดลงคอมพิวเตอร์ของคุณทันที (.download)
    if USING_COLAB:
        print(f"📥 กำลังดาวน์โหลดไฟล์: {zip_filename} ลงเครื่องคอมพิวเตอร์ของคุณ...")
        files.download(zip_filename)
    else:
        print(f"💾 บันทึกไฟล์ลงเครื่องสำเร็จในโฟลเดอร์ปัจจุบัน: {zip_filename}")




Map

ดาวน์โหลดลงคอม

In [ ]:
# ==============================================================================
# STEP 7: [ปรับปรุง] รันระบบและดาวน์โหลดอัตโนมัติรวดเดียว 3 ไฟล์
# ==============================================================================

print("⏳ ระบบกำลังประมวลผลข้อมูล และจะทำการดาวน์โหลดไฟล์ลงคอมพิวเตอร์ให้โดยอัตโนมัติ...")
print("-" * 70)

# ช่วงที่ 1: ปี 1993 - 2003 (ไฟล์จะเด้งโหลดอัตโนมัติ)
export_display_and_download_direct(change_93_03, 'Erosion_Change_1993_2003', '(1993-2003)')

# ช่วงที่ 2: ปี 2003 - 2026 (ไฟล์จะเด้งโหลดอัตโนมัติ)
export_display_and_download_direct(change_03_26, 'Erosion_Change_2003_2026', '(2003-2026)')

# ช่วงที่ 3: ภาพรวมปี 1993 - 2026 (ไฟล์จะเด้งโหลดอัตโนมัติ)
export_display_and_download_direct(change_93_26, 'Erosion_Change_Total_1993_2026', 'ภาพรวม (1993-2026)')

print("-" * 70)
print("✅ เสร็จสิ้นกระบวนการ ดาวน์โหลดไฟล์ .zip เรียบร้อยแล้ว!")

⏳ ระบบกำลังประมวลผลข้อมูล และจะทำการดาวน์โหลดไฟล์ลงคอมพิวเตอร์ให้โดยอัตโนมัติ...
----------------------------------------------------------------------
📥 กำลังดาวน์โหลดไฟล์: Erosion_Change_1993_2003.zip ลงเครื่องคอมพิวเตอร์ของคุณ...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>